<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Tabnet_Cow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

In [20]:
cow='/content/rural_carbon_dataset1.csv'
cow_df=pd.read_csv(cow)
cow_df.head()


,Region,Month,Fertilizer_Usage_kg,Crop_Type,Crop_Area_ha,Livestock_Cows,Livestock_Pigs,Household_Energy_kWh,Renewable_Energy_Fraction,Temperature_C,Rainfall_mm,Carbon_Emission_tCO2,Year,classes
0,R0103,4,106.5,Maize,37.3,0,130,296.6,0.58,23.6,70.7,5.91,2024,Low
1,R0271,11,100.9,Wheat,47.4,95,196,408.8,0.50,24.9,275.5,21.63,2021,High
2,R0107,7,59.4,Maize,28.7,83,120,479.3,0.97,21.8,35.7,9.24,2022,Low
3,R0072,8,122.9,Rice,29.3,53,82,146.3,0.13,30.7,285.5,3.71,2024,Low
4,R0189,2,66.4,Maize,25.7,46,169,233.9,0.90,23.3,76.2,4.37,2020,Low


In [10]:
cow_df=cow_df.drop(['Region','Year','Month','Carbon_Emission_tCO2'], axis=1)
# Separate target from predictors
target = cow_df['classes']
Features = cow_df.drop(columns=['classes'])

categorical_columns = [col for col in Features.columns if Features[col].dtype == 'object' or Features[col].dtype.name == 'category']
cat_idxs = [i for i, f in enumerate(Features.columns) if f in categorical_columns]

X_train_df, X_tmp_df, y_train, y_tmp = train_test_split(Features, target.values, test_size=0.3, random_state=42, stratify=target.values)
X_valid_df, X_test_df, y_valid, y_test = train_test_split(X_tmp_df, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

cat_dims = []
X_train = X_train_df.copy()
X_valid = X_valid_df.copy()
X_test = X_test_df.copy()

for col in categorical_columns:
    l_enc = LabelEncoder()

    # الـ fit يتم على بيانات التدريب فقط
    X_train[col] = l_enc.fit_transform(X_train_df[col].values)

    # الـ transform يتم على الباقي مع التعامل مع أي قيم جديدة لم تظهر في التدريب
    # (تم تحويل القيم غير المعروفة إلى -1 تفادياً للمشاكل)
    X_valid[col] = X_valid_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)
    X_test[col] = X_test_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)

    # حساب الأبعاد (نضيف +1 لحساب القيم غير المعروفة -1 إن وجدت)
    cat_dims.append(len(l_enc.classes_) + 1)

# 6. تحويل جداول ميزات البيانات إلى مصفوفات Numpy جاهزة للـ TabNet
X_train = X_train.values
X_valid = X_valid.values
X_test = X_test.values

print("تم تجهيز البيانات وحل مشكلة التسريب بنجاح!")
print(f"أبعاد الأعمدة الفئوية (cat_dims): {cat_dims}")
print(f"مواقع الأعمدة الفئوية (cat_idxs): {cat_idxs}")




تم تجهيز البيانات وحل مشكلة التسريب بنجاح!
أبعاد الأعمدة الفئوية (cat_dims): [5]
مواقع الأعمدة الفئوية (cat_idxs): [1]


In [14]:
# 5. تعريف نموذج TabNet
clf = TabNetClassifier(
    cat_idxs=cat_idxs,       # مواقع الأعمدة الفئوية
    cat_dims=cat_dims,       # أبعاد (عدد تصنيفات) كل عمود فئوي
    cat_emb_dim=1,           # حجم الـ Embedding لكل متغير فئوي (يمكن زيادته إذا كانت التصنيفات كثيرة)
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":50, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax'    # يعطي تفسيرية أفضل (Interpretability) عبر تحديد الخصائص بوضوح
   )

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [15]:
# 6. تدريب النموذج
clf.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    eval_name=['train', 'valid'],
    eval_metric=['accuracy', 'logloss'], # المقاييس المستخدمة لمراقبة الأداء
    max_epochs=100,
    patience=15,                    # إيقاف مبكر (Early Stopping) إذا لم يتحسن الأداء لـ 15 دورة
    batch_size=128,
    virtual_batch_size=16,          # تقنية مستخدمة في TabNet لتحسين استقرار الـ Batch Normalization
    num_workers=0,
    drop_last=False
)

epoch 0  | loss: 1.32196 | train_accuracy: 0.44095 | train_logloss: 3.24316 | valid_accuracy: 0.42444 | valid_logloss: 3.09507 |  0:00:00s
epoch 1  | loss: 1.05915 | train_accuracy: 0.4919  | train_logloss: 2.38332 | valid_accuracy: 0.48667 | valid_logloss: 2.54611 |  0:00:01s
epoch 2  | loss: 1.00666 | train_accuracy: 0.42762 | train_logloss: 1.84099 | valid_accuracy: 0.43778 | valid_logloss: 1.85981 |  0:00:02s
epoch 3  | loss: 0.97206 | train_accuracy: 0.50952 | train_logloss: 1.80024 | valid_accuracy: 0.5     | valid_logloss: 1.76252 |  0:00:03s
epoch 4  | loss: 0.98845 | train_accuracy: 0.52048 | train_logloss: 1.16353 | valid_accuracy: 0.53333 | valid_logloss: 1.2016  |  0:00:04s
epoch 5  | loss: 0.96738 | train_accuracy: 0.51667 | train_logloss: 1.13564 | valid_accuracy: 0.53333 | valid_logloss: 1.1556  |  0:00:05s
epoch 6  | loss: 0.94614 | train_accuracy: 0.4881  | train_logloss: 1.08564 | valid_accuracy: 0.50889 | valid_logloss: 1.08744 |  0:00:06s
epoch 7  | loss: 0.94    | 

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [16]:
# 7. التقييم والتنبؤ بالبيانات الجديدة
predictions = clf.predict(X_test)
print(f"Test Accuracy: {np.mean(predictions == y_test):.4f}")

Test Accuracy: 0.5978


In [18]:
# معرفة الأهمية النسبية لكل عمود في الداتا سيت بالكامل
feature_importances = clf.feature_importances_

for feature_name, importance in zip(Features, feature_importances):
    print(f"Feature: {feature_name} -> Importance: {importance:.4f}")

# استخراج الـ Mask (قناع الانتباه) لرؤية الأجزاء التي ركز عليها النموذج في عينة معينة
explain_matrix, masks = clf.explain(X_test)
# يمكنك رسم الـ masks هنا باستخدام matplotlib لرؤية الأوزان بصرياً

Feature: Fertilizer_Usage_kg -> Importance: 0.0204
Feature: Crop_Type -> Importance: 0.0003
Feature: Crop_Area_ha -> Importance: 0.1231
Feature: Livestock_Cows -> Importance: 0.3838
Feature: Livestock_Pigs -> Importance: 0.2056
Feature: Household_Energy_kWh -> Importance: 0.0235
Feature: Renewable_Energy_Fraction -> Importance: 0.1806
Feature: Temperature_C -> Importance: 0.0622
Feature: Rainfall_mm -> Importance: 0.0007


In [23]:
cow_2=cow_df.drop(['Region','Year','Month','Carbon_Emission_tCO2'], axis=1)

# 4. Feature Engineering
cow_weight = 2.0  # Approx tons of CO2e per cow/year
pig_weight = 0.3  # Approx tons of CO2e per pig/year

cow_2['Weighted_Livestock_Impact'] = (cow_2['Livestock_Cows'] * cow_weight) + (cow_df['Livestock_Pigs'] * pig_weight)
# 2. Advanced Density Feature: Connect the weighted impact with the land area
# cow_df['Livestock_Impact_per_Ha'] = cow_df['Weighted_Livestock_Impact'] / (cow_df['Crop_Area_ha'] + 1e-5)

cow_2['Fert_x_Area'] = cow_2['Fertilizer_Usage_kg'] / cow_2['Crop_Area_ha']
cow_2['Renew_x_Energy'] = cow_2['Renewable_Energy_Fraction'] * cow_2['Household_Energy_kWh']
cow_2['energy_intensity'] = cow_2['Household_Energy_kWh'] / (cow_2['Crop_Area_ha'] + 1e-5)
cow_2=cow_2.drop(['Livestock_Pigs','Livestock_Cows','Renewable_Energy_Fraction','Fertilizer_Usage_kg'], axis=1)

# print a sample of dataset after preprocessing
cow_2.head()

,Crop_Type,Crop_Area_ha,Household_Energy_kWh,Temperature_C,Rainfall_mm,classes,Weighted_Livestock_Impact,Fert_x_Area,Renew_x_Energy,energy_intensity
0,Maize,37.3,296.6,23.6,70.7,Low,39.0,2.855228,172.028,7.951740
1,Wheat,47.4,408.8,24.9,275.5,High,248.8,2.128692,204.400,8.624471
2,Maize,28.7,479.3,21.8,35.7,Low,202.0,2.069686,464.921,16.700343
3,Rice,29.3,146.3,30.7,285.5,Low,130.6,4.194539,19.019,4.993172
4,Maize,25.7,233.9,23.3,76.2,Low,142.7,2.583658,210.510,9.101164


In [25]:
target2 = cow_2['classes']
Features2 = cow_2.drop(columns=['classes'])

categorical2_columns = [col for col in Features2.columns if Features2[col].dtype == 'object' or Features2[col].dtype.name == 'category']
cat2_idxs = [i for i, f in enumerate(Features2.columns) if f in categorical_columns]

X2_train_df, X2_tmp_df, y2_train, y2_tmp = train_test_split(Features2, target2.values, test_size=0.3, random_state=42, stratify=target2.values)
X2_valid_df, X2_test_df, y2_valid, y2_test = train_test_split(X2_tmp_df, y2_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

cat2_dims = []
X2_train = X2_train_df.copy()
X2_valid = X2_valid_df.copy()
X2_test = X2_test_df.copy()

for col in categorical2_columns:
    l_enc = LabelEncoder()

    # الـ fit يتم على بيانات التدريب فقط
    X2_train[col] = l_enc.fit_transform(X2_train_df[col].values)

    # الـ transform يتم على الباقي مع التعامل مع أي قيم جديدة لم تظهر في التدريب
    # (تم تحويل القيم غير المعروفة إلى -1 تفادياً للمشاكل)
    X2_valid[col] = X2_valid_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)
    X2_test[col] = X2_test_df[col].map(lambda s: l_enc.transform([s])[0] if s in l_enc.classes_ else -1)

    # حساب الأبعاد (نضيف +1 لحساب القيم غير المعروفة -1 إن وجدت)
    cat2_dims.append(len(l_enc.classes_) + 1)

# 6. تحويل جداول ميزات البيانات إلى مصفوفات Numpy جاهزة للـ TabNet
X2_train = X2_train.values
X2_valid = X2_valid.values
X2_test = X2_test.values

print("تم تجهيز البيانات وحل مشكلة التسريب بنجاح!")
print(f"أبعاد الأعمدة الفئوية (cat_dims2): {cat_dims}")
print(f"مواقع الأعمدة الفئوية (cat_idxs2): {cat_idxs}")


تم تجهيز البيانات وحل مشكلة التسريب بنجاح!
أبعاد الأعمدة الفئوية (cat_dims2): [5]
مواقع الأعمدة الفئوية (cat_idxs2): [1]


In [26]:
# 5. تعريف نموذج TabNet
clf2 = TabNetClassifier(
    cat_idxs=cat2_idxs,       # مواقع الأعمدة الفئوية
    cat_dims=cat2_dims,       # أبعاد (عدد تصنيفات) كل عمود فئوي
    cat_emb_dim=1,           # حجم الـ Embedding لكل متغير فئوي (يمكن زيادته إذا كانت التصنيفات كثيرة)
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":50, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax'    # يعطي تفسيرية أفضل (Interpretability) عبر تحديد الخصائص بوضوح
   )

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [28]:
# 6. تدريب النموذج
clf2.fit(
    X_train=X2_train, y_train=y2_train,
    eval_set=[(X2_train, y2_train), (X2_valid, y2_valid)],
    eval_name=['train', 'valid'],
    eval_metric=['accuracy', 'logloss'], # المقاييس المستخدمة لمراقبة الأداء
    max_epochs=100,
    patience=15,                    # إيقاف مبكر (Early Stopping) إذا لم يتحسن الأداء لـ 15 دورة
    batch_size=128,
    virtual_batch_size=16,          # تقنية مستخدمة في TabNet لتحسين استقرار الـ Batch Normalization
    num_workers=0,
    drop_last=False
)

epoch 0  | loss: 1.37294 | train_accuracy: 0.41095 | train_logloss: 2.63698 | valid_accuracy: 0.43778 | valid_logloss: 2.55947 |  0:00:00s
epoch 1  | loss: 1.11591 | train_accuracy: 0.43095 | train_logloss: 1.62912 | valid_accuracy: 0.47111 | valid_logloss: 1.62606 |  0:00:01s
epoch 2  | loss: 1.04633 | train_accuracy: 0.45238 | train_logloss: 1.45465 | valid_accuracy: 0.48889 | valid_logloss: 1.42549 |  0:00:02s
epoch 3  | loss: 1.01269 | train_accuracy: 0.52619 | train_logloss: 1.20125 | valid_accuracy: 0.53111 | valid_logloss: 1.24304 |  0:00:03s
epoch 4  | loss: 1.01503 | train_accuracy: 0.50905 | train_logloss: 1.13283 | valid_accuracy: 0.51111 | valid_logloss: 1.15224 |  0:00:03s
epoch 5  | loss: 0.99714 | train_accuracy: 0.51952 | train_logloss: 1.12736 | valid_accuracy: 0.51556 | valid_logloss: 1.15725 |  0:00:04s
epoch 6  | loss: 0.9784  | train_accuracy: 0.50905 | train_logloss: 1.07945 | valid_accuracy: 0.46    | valid_logloss: 1.10938 |  0:00:05s
epoch 7  | loss: 0.97593 | 

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [29]:
# 7. التقييم والتنبؤ بالبيانات الجديدة
predictions = clf2.predict(X2_test)
print(f"Test Accuracy: {np.mean(predictions == y2_test):.4f}")

Test Accuracy: 0.5467


In [30]:
# معرفة الأهمية النسبية لكل عمود في الداتا سيت بالكامل
feature_importances2 = clf2.feature_importances_

for feature_name, importance in zip(Features2, feature_importances2):
    print(f"Feature: {feature_name} -> Importance: {importance:.4f}")

# استخراج الـ Mask (قناع الانتباه) لرؤية الأجزاء التي ركز عليها النموذج في عينة معينة
explain_matrix, masks = clf2.explain(X2_test)
# يمكنك رسم الـ masks هنا باستخدام matplotlib لرؤية الأوزان بصرياً

Feature: Crop_Type -> Importance: 0.0054
Feature: Crop_Area_ha -> Importance: 0.0750
Feature: Household_Energy_kWh -> Importance: 0.1153
Feature: Temperature_C -> Importance: 0.0143
Feature: Rainfall_mm -> Importance: 0.0579
Feature: Weighted_Livestock_Impact -> Importance: 0.3034
Feature: Fert_x_Area -> Importance: 0.0577
Feature: Renew_x_Energy -> Importance: 0.3232
Feature: energy_intensity -> Importance: 0.0479
